# Silicon Traveling-Wave Mach-Zehnder Modulator: from Doping to Bandwidth

A silicon modulator writes an electrical signal onto light. This notebook
simulates one end to end, starting from a drawn gdsfactory component and ending
with the numbers a designer quotes: **modulation efficiency** $V_\pi L$,
**optical loss**, **line impedance** and **electro-optic bandwidth**. Everything
runs on open-source solvers.

## How the device works

- The waveguide is a silicon rib with a **PN junction** along its centre. Around
  the junction sits a *depletion region*, a strip of silicon emptied of free
  electrons and holes.
- A **reverse bias** widens that depleted strip. Fewer free carriers means a
  slightly higher refractive index and a slightly lower absorption: this is the
  *plasma-dispersion effect*. The light in the rib therefore picks up a
  voltage-controlled phase.
- A **Mach-Zehnder interferometer** turns the phase shift into an intensity
  change. $V_\pi L$ is the voltage needed for a $\pi$ phase shift, times the
  length it takes; lower is better.
- The effect is weak ($\Delta n_{eff} \sim 10^{-4}$), so the phase shifter is
  millimetres long, which is many RF wavelengths at tens of GHz. The drive
  electrode is therefore a **transmission line** running alongside the waveguide,
  so that the RF wave travels *with* the light: a *traveling-wave* modulator.
  Bandwidth is then set by three things: how well the RF and optical velocities
  match, how much the RF wave is attenuated along the line, and how well the
  line is matched to its source and termination.

## What the notebook computes

| Step | Question | Solver | Result |
|---|---|---|---|
| 1. Charge | Where are the electrons and holes at each bias? | DEVSIM (drift-diffusion) | carrier maps $n(y,z)$, $p(y,z)$, junction capacitance $C(V)$ |
| 2. Carriers | What do those carriers do to the silicon? | closed-form | optical $\Delta n$, $\Delta\alpha$; RF conductivity $\sigma$ |
| 3. Optical | How does the guided mode respond? | femwell (FEM mode solver) | $n_{eff}(V)$, optical loss, $V_\pi L$ |
| 4. RF | What transmission line does the RF signal see? | femwell (or Palace) | RF index, RF loss, $Z_0$ versus frequency |
| 5. Line | What does the whole device do? | transmission-line model | EO response, 3 dB bandwidth, RLGC |

All the physics is solved on the **cross-section** of the device: the $(y, z)$
plane at $x = 0$, with the light and the RF wave propagating along $x$. $y$ runs
across the junction and $z$ is vertical.

Install everything with the combined extra:

```bash
pip install 'gsim[modulator]'
```

## The device, and a Study over it

The example is a depletion-mode phase shifter as foundries build one: a
500 × 220 nm silicon rib on a 90 nm slab, with a lateral PN junction at the
centre of the rib. Going out from the junction on each side, the silicon is
`rib | slab | plus | contact`:

- the lightly doped **core** (p 5e17, n 3e17 cm⁻³): the rib half and 0.5 µm of
  slab beside it. This is where the light is and where the depletion edge
  moves. Light doping keeps the optical loss down;
- a moderately doped **plus** region (1e19 cm⁻³, 1 µm wide) that carries the
  current to the core with little resistance;
- a heavily doped **contact** region (1e20 cm⁻³, 1 µm wide) under the metal, so
  the metal-silicon contact is Ohmic.

The doping increases in steps away from the rib because free carriers absorb
light: you want them near the metal, where they lower the series resistance,
and not near the optical mode. The traveling-wave electrodes are 10 µm wide,
1 µm thick aluminium.

These are generic published values for a 220 nm SOI depletion modulator, not any
foundry's. `rib_phase_shifter()` draws this device so the notebook has something
to work on. **In your own workflow you bring your own component, layer stack and
device description**, and everything below stays the same.

### Study and Stages

`pn_phase_shifter` returns a **Study**: one object holding the device and the
five steps of the table above, called **Stages**: `study.charge`,
`study.carriers`, `study.optical`, `study.rf` and `study.line`.

- **Configure** a Stage by calling it with settings: `study.rf(n_strips=41)`.
- **Run** it with `.run()`, which returns its result. A Stage first runs
  whatever it depends on, so `study.report()` alone runs the full chain.
- Results are **cached**. Changing a setting clears that Stage's result and
  those of the Stages that depend on it, and nothing else, so a parameter
  study re-solves only what it has to.

The device description only says which regions are p and which are n. From that
and the drawn cross-section the Study works out the rest: where the metal
contacts are, where the junction is, and which part of the cross-section each
Stage needs to mesh. The printout below shows what it found.

In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np

from gsim.common.stack.materials import MATERIALS_DB
from gsim.modulator import pn_phase_shifter, rib_phase_shifter

# The mesher's advice on air boxes and absorbing boundaries targets 3D
# driven simulations; it does not apply to these cross-section solves.
logging.getLogger("gsim.palace").setLevel(logging.ERROR)

WAVELENGTH_UM = 1.55
REVERSE_BIASES = np.linspace(0, 2, 11)  # V on the cathode = reverse bias
RF_FREQS_HZ = [10e9, 20e9, 40e9, 60e9, 80e9, 100e9]
LENGTH_UM = 3000.0  # traveling-wave electrode length

demo = rib_phase_shifter()  # your own component and stack go here

# The demo draws its doped silicon at a constant permittivity. Give it
# silicon's own dispersion at optical wavelengths, so the group index computed
# in Step 5 carries material dispersion and not only the waveguide's.
sellmeier = next(
    m for m in MATERIALS_DB["silicon"].dispersion_models if m.type == "sellmeier"
)
for name in demo.device.doped_regions:
    demo.stack.materials[name].setdefault("dispersion_models", []).append(
        sellmeier.model_dump()
    )

study = pn_phase_shifter(
    component=demo.component,
    stack=demo.stack,
    device=demo.device,
    electrodes=demo.electrodes,
    biases=REVERSE_BIASES,
    wavelength_um=WAVELENGTH_UM,
    frequencies_hz=RF_FREQS_HZ,
    length_um=LENGTH_UM,
    output_dir="./tcad-twmzm",
    verbose=False,  # True prints each Stage's timing and the solver logs
)

# The bare electrode of the segmented-line section has two slow modes close
# together; a wider mode search tells them apart. Set here so the loaded line
# is solved once.
study.rf(num_modes=8, degeneracy_rtol=0.01)

layout = study.layout
print("doping (cm^-3):", {name: f"{c:.0e}" for name, c in demo.doping_cm3.items()})
print("contacts:     ", [(c.name, c.electrode, c.region) for c in layout.contacts])
print("interfaces:   ", [i.name for i in layout.interfaces])
print(f"junction:      {layout.junction.regions} at y = {layout.junction_position} um")

### One component, three cross-section windows

Each Stage meshes only the part of the cross-section its physics needs, its
**window**, and works that window out from the device by itself:

- **charge** spans the doped silicon from one metal contact to the other. The
  oxide far away and the bulk of the electrodes play no role in carrier
  transport;
- **optical** is a box of a few µm around the rib, enough to hold the guided
  mode and its evanescent tails;
- **RF** keeps the full cross-section, because the RF field fills the whole gap
  between the electrodes and the space around them.

In [ ]:
optical_window = layout.window_around_junction(margin_um=study.optical.mode_margin_um)
optical_window_z = layout.window_z_around_guide(
    above_um=study.optical.z_above_um, below_um=study.optical.z_below_um
)
print(f"charge  window: y in {layout.window} um (doped slab between contacts)")
print(f"optical window: y in {optical_window} um, z in {optical_window_z} um")
print(f"rf      window: {study.rf.window} (None = the full cross-section)")
print(f"rf strips tile: y in {study.rf.strip_span} um (the doped slab, pads included)")

### A helper to draw each Stage's mesh

Every Stage writes its mesh into its own folder under `output_dir`. The helper
below draws one, with a colour per region, or a colour scale for regions given
a value (used later for conductivity). Metal that a Stage leaves out of its
mesh shows as a white hole. It is plotting code only; skip it on a first read.

In [ ]:
import meshio
from matplotlib.collections import PolyCollection
from matplotlib.colors import LogNorm

REGION_COLORS = {}  # one colour per region name, stable across figures


def plot_cross_section(
    mesh_path,
    *,
    ax,
    title=None,
    values=None,
    label=None,
    limits=None,
    xlim=None,
    ylim=None,
):
    """Draw a Stage's cross-section mesh, one colour per region.

    Regions named in ``values`` (region -> number) are coloured by that
    number on a log scale instead, clipped below at 1e-2, spanning
    ``limits`` when given so two figures can share one scale.
    """
    mesh = meshio.read(str(mesh_path))
    points = mesh.points[:, :2]
    names = {tag: name for name, (tag, dim) in mesh.field_data.items() if dim == 2}
    triangles = mesh.cells_dict["triangle"]
    tags = mesh.cell_data_dict["gmsh:physical"]["triangle"]

    handles, scalar_polys, scalar_values = [], [], []
    for tag in sorted(set(tags), key=names.get):
        name, polys = names[tag], points[triangles[tags == tag]]
        if values is not None and name in values:
            scalar_polys.extend(polys)
            scalar_values.extend([values[name]] * len(polys))
            continue
        color = REGION_COLORS.setdefault(
            name, plt.get_cmap("tab20")(len(REGION_COLORS) % 20)
        )
        ax.add_collection(
            PolyCollection(polys, facecolors=color, edgecolors="k", linewidths=0.1)
        )
        handles.append(plt.Rectangle((0, 0), 1, 1, color=color, label=name))
    if scalar_polys:
        shown = np.maximum(np.asarray(scalar_values, dtype=float), 1e-2)
        low, high = limits or (shown.min(), max(shown.max(), 10 * shown.min()))
        coll = PolyCollection(
            scalar_polys,
            array=shown,
            cmap="viridis",
            norm=LogNorm(low, high),
            edgecolors="k",
            linewidths=0.1,
        )
        ax.add_collection(coll)
        plt.colorbar(coll, ax=ax, label=label, fraction=0.03)

    ax.autoscale_view()
    ax.set_aspect("equal")
    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.set(xlabel="y (um)", ylabel="z (um)", title=title)
    if handles:
        ax.legend(
            handles=handles,
            fontsize=7,
            ncol=len(handles),
            loc="upper center",
            bbox_to_anchor=(0.5, -0.3),
        )
    return ax

## Step 1: charge transport (DEVSIM)

This Stage solves the semiconductor equations on the doped silicon: Poisson's
equation for the electrostatic potential, coupled to drift-diffusion continuity
equations for electrons and holes. It does so at every bias of the sweep.

The bias is applied to the **cathode** (the contact on the n side), so a
positive voltage is a *reverse* bias: the regime a depletion modulator is
driven in, where almost no current flows and the junction behaves as a
voltage-dependent capacitor.

Each bias point gives:

- the **carrier maps** $n(y,z)$ and $p(y,z)$, which every later step uses;
- the **terminal currents** (tiny leakage, in reverse bias);
- the **junction capacitance** per unit length, from a small-signal AC solve
  around that bias.

The depletion edge moves by only tens of nanometres over the sweep, so the Stage
holds its mesh to 10 nm across the whole rib, where that edge moves, and not
only along the junction line. It also solves Poisson's equation in the oxide
around the silicon (the potential alone: there are no carriers there), so that
the capacitance counts the field that fringes around the junction. Expect this
step to take a minute or two.

In [ ]:
sweep = study.charge.run()

for point in sweep.points:
    print(
        f"V = {point.bias_v:4.1f} V:  C = {point.capacitance_f_per_m * 1e12:6.1f} pF/m,"
        f"  I_cathode = {point.currents_a_per_cm['cathode']:+.2e} A/cm"
    )

The capacitance falls as the reverse bias grows: the depletion region is the
capacitor's dielectric, and it is getting wider.

Below is the mesh this Stage solved on: the eight doped regions between the two
contacts, surrounded by oxide. The electrodes are not part of the semiconductor
problem. They appear as white holes, and the edges where they touch the
silicon are the `anode` and `cathode` contacts where the voltage is applied.
The close-up shows the 10 nm elements across the rib.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), height_ratios=(1, 2))
charge_mesh = study.stage_dir("charge") / "palace.msh"
junction_y = layout.junction_position
plot_cross_section(
    charge_mesh, ax=axes[0], title="Charge window: the doped slab", ylim=(-0.3, 1.2)
)
axes[0].get_legend().remove()  # the close-up below carries it
plot_cross_section(
    charge_mesh,
    ax=axes[1],
    title="Around the junction",
    xlim=(junction_y - 0.3, junction_y + 0.3),
    ylim=(-0.05, 0.3),
)
plt.tight_layout()
plt.show()

### Check: $C(V)$ against the textbook formula

For an abrupt junction the textbook depletion approximation gives the
depletion width $W(V)$ in closed form, and the capacitance per unit length is
that of a parallel-plate capacitor, $C = \varepsilon_{Si}\, h / W(V)$, with $h$
the rib height. `PNJunctionConfig` implements it, and
`tcad.compare_capacitance` puts it next to the simulated curve.

The simulation sits 106 to 122 pF/m above the formula at every bias. The
formula is one-dimensional: it is the path through the depleted silicon alone.
The offset is the field that fringes through the oxide above and below the
junction, from one conducting half of the silicon to the other: a second
capacitor in parallel, whose plates hardly move with the bias, which is why the
offset is nearly constant while the relative deviation grows along the sweep.
With `study.charge(oxide=False)` the solve keeps to the silicon, and the two
agree to 3 %.

In [ ]:
from gsim import tcad
from gsim.common.stack.pn_junction import PNJunctionConfig

junction = PNJunctionConfig(
    na_cm3=study.device.p_doping_cm3, nd_cm3=study.device.n_doping_cm3
)
comparison = tcad.compare_capacitance(junction, sweep, height_um=demo.rib_height_um)
offset = (comparison.c_tcad_f_per_cm - comparison.c_analytic_f_per_cm) * 1e14
print(f"max relative deviation: {comparison.max_relative_deviation:.1%}")
print(f"offset above the formula: {offset.min():.0f} to {offset.max():.0f} pF/m")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(
    comparison.v_reverse, comparison.c_tcad_f_per_cm * 1e14, "o-", label="TCAD (DEVSIM)"
)
ax.plot(
    comparison.v_reverse,
    comparison.c_analytic_f_per_cm * 1e14,
    "s--",
    label="Analytic (Sze)",
)
ax.set_xlabel("Reverse bias (V)")
ax.set_ylabel("C (fF per um of length)")
ax.set_title("Junction capacitance: TCAD vs depletion approximation")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

### Carrier maps

The electron concentration across the rib at the two ends of the sweep, on a
log scale. The n side is on the left (high $n$), the p side on the right (almost
no electrons), and the transition between them is the depletion edge. At 2 V it
has moved further into the n side: that displacement, a few tens of
nanometres, is the entire modulation mechanism.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3), sharey=True)
for ax, point in zip(axes, (sweep.points[0], sweep.points[-1]), strict=False):
    carriers = point.carriers
    tri = ax.tricontourf(
        carriers.x_um,
        carriers.y_um,
        np.log10(np.maximum(carriers.electrons_cm3, 1.0)),
        levels=30,
    )
    ax.set_title(f"Electrons at V = {point.bias_v:g} V")
    ax.set_xlabel("y (um)")
    ax.set_xlim(*layout.junction_span.h)  # the rib
fig.colorbar(tri, ax=axes, label="log10 n (cm^-3)")
axes[0].set_ylabel("z (um)")
plt.show()

## Step 2: from carriers to material properties

Free carriers change the silicon in two ways, and this Stage evaluates both at
every mesh node of every bias point:

- **At optical frequencies**, the plasma-dispersion effect: an index change
  $\Delta n$ (negative: carriers lower the index) and an absorption
  $\Delta\alpha$. These are empirical power laws in $n$ and $p$; holes change
  the index more than electrons and absorb less for the same index change,
  which is why the junction is usually offset to put more of the mode on the
  p side.
- **At RF**, a conductivity $\sigma = q(\mu_n n + \mu_p p)$: doped silicon is a
  (poor) conductor, a depleted region is an insulator. The mobilities $\mu$
  fall with doping, because carriers scatter off the dopant atoms: the contact
  regions are doped 300 times more heavily than the core but conduct only
  about 40 times better.

The default optical coefficients are the published silicon fits of Nedeljkovic
et al. (2011), chosen to match the wavelength (1.55 or 1.31 µm);
foundry-calibrated values go in with `study.carriers(dispersion=...)`. The
mobility model is the published silicon fit of Masetti et al. (1983). It is
shared with the charge simulation, so that both steps agree on the resistance
of the silicon; change it with `study.charge(mobility=...)`.

The plots are a horizontal cut through the middle of the rib. At the centre,
where the silicon is depleted, all three quantities drop to zero, and that
region is wider at 2 V.

In [ ]:
response = study.carriers.run()
print("coefficients fitted at:", study.carriers.dispersion.wavelength_um, "um")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for point in (response.points[0], response.points[-1]):
    carriers = point.carriers
    band = np.abs(carriers.y_um - demo.rib_height_um / 2) < 0.03
    order = np.argsort(carriers.x_um[band])
    y = carriers.x_um[band][order]  # the map's first coordinate is device y
    label = f"V = {point.bias_v} V"
    axes[0].plot(y, point.index_shift[band][order], label=label)
    axes[1].plot(y, point.absorption_cm[band][order], label=label)
    axes[2].plot(y, point.conductivity_s_per_m[band][order], label=label)
axes[0].set_ylabel("$\\Delta n$")
axes[1].set_ylabel("$\\Delta\\alpha$ (1/cm)")
axes[2].set_ylabel("$\\sigma$ (S/m)")
for ax in axes:
    ax.set_xlabel("y (um)")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.suptitle("Carrier-derived material response across the junction (mid-rib cut)")
plt.tight_layout()
plt.show()

## Step 3: the optical mode at each bias

The optical Stage meshes its own window around the rib, interpolates each bias
point's carrier map onto that mesh, and solves for the guided mode with a
position-dependent complex permittivity: each element of doped silicon takes
$\varepsilon = (n_{Si} + \Delta n - i\kappa)^2$, with $\kappa$ from the local
absorption. One complex effective index comes out per bias:

- the change of its real part with bias, $\Delta n_{eff}(V)$, is the phase
  modulation;
- its imaginary part is the optical loss caused by the free carriers.

In [ ]:
optical = study.optical.run()

for mode in optical.points:
    print(
        f"V = {mode.bias_v:4.1f} V:  n_eff = {mode.n_eff:.6f},  "
        f"dn_eff = {mode.index_shift:+.2e},  loss = {mode.loss_db_cm:5.2f} dB/cm"
    )

As the bias rises, carriers leave the centre of the mode: $n_{eff}$ goes up and
the loss goes down. Note the scale: the index moves in the fifth decimal place,
which is why the mode solve needs the carrier map resolved rather than averaged.

The optical mesh, for comparison with the charge mesh above: a smaller window,
and coarser elements, since it resolves a wavelength-scale mode rather than a
nanometre-scale depletion edge.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
plot_cross_section(
    study.stage_dir("optical") / "palace.msh", ax=ax, title="Optical window"
)
plt.tight_layout()
plt.show()

### Modulation efficiency $V_\pi L$ and optical loss

A phase shift of $\pi$ over a length $L$ needs $\Delta n_{eff} = \lambda / 2L$,
so the efficiency follows from the local slope of the curve on the left:

$$V_\pi L = \frac{\lambda}{2\,\left|\mathrm{d}n_{eff}/\mathrm{d}V\right|}$$

$V_\pi L$ rises with bias because the depletion width grows like $\sqrt{V}$:
each extra volt moves the edge a little less than the previous one. The value
is quoted one point before the end of the sweep, where the slope is a centred
difference. About 2 V·cm is typical for this kind of junction.

There is a trade-off here. Heavier core doping improves $V_\pi L$ and
worsens the loss; the product of the two is the usual figure of merit when
comparing junction designs.

In [ ]:
from gsim.modulator.twmzm import vpi_length_vcm

vpi_l = vpi_length_vcm(
    optical.voltages, optical.index_shift, wavelength_um=optical.wavelength_um
)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
axes[0].plot(optical.voltages, optical.index_shift * 1e5, "o-")
axes[0].set_ylabel("$\\Delta n_{eff} \\times 10^{-5}$")
axes[1].plot(optical.voltages, vpi_l, "o-")
axes[1].set_ylabel("$V_\\pi L$ (V cm)")
axes[2].plot(optical.voltages, optical.loss_db_cm, "o-")
axes[2].set_ylabel("optical loss (dB/cm)")
for ax in axes:
    ax.set_xlabel("Reverse bias (V)")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f"V_pi L at {optical.voltages[-2]:.1f} V: {vpi_l[-2]:.2f} V cm")

### What a graded junction costs

So far every region is doped at one level and the junction is perfectly abrupt.
A real implant is not: the dopants scatter sideways as they stop (*lateral
straggle*) and diffuse during the anneal, so each doping step is smeared over
some tens of nanometres. Donors then reach into the p side and acceptors into
the n side. Where both are present they **compensate**: only their difference
dopes the silicon, so the net doping near the junction is lower than drawn.

`rib_phase_shifter(lateral_straggle_um=...)` draws the same device with every
doping step smeared into an error function of that standard deviation. Nothing
else changes: the graded profiles go into the same device description, and the
Study runs on it as before. The cell below solves the charge and optical Stages
for a 30 nm straggle, on the same mesh, which takes as long again as the
abrupt device did.

In [ ]:
STRAGGLE_UM = 0.03  # lateral straggle of every doping step

graded_demo = rib_phase_shifter(lateral_straggle_um=STRAGGLE_UM)
graded_study = pn_phase_shifter(
    component=graded_demo.component,
    stack=graded_demo.stack,
    device=graded_demo.device,
    electrodes=graded_demo.electrodes,
    biases=REVERSE_BIASES,
    wavelength_um=WAVELENGTH_UM,
    output_dir="./tcad-twmzm-graded",
    verbose=False,
)
graded_sweep = graded_study.charge.run()
graded_optical = graded_study.optical.run()
graded_vpi_l = vpi_length_vcm(
    graded_optical.voltages,
    graded_optical.index_shift,
    wavelength_um=graded_optical.wavelength_um,
)

In [ ]:
from gsim.tcad.doping import acceptor_donor_concentrations

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))

# The doping across the rib. Each profile belongs to one region, so the two
# halves of the rib are evaluated separately.
y = np.linspace(*layout.junction_span.h, 400)
for name in layout.junction.regions:
    low, high = layout.region_spans[name].h
    inside = (y >= low) & (y <= high)
    for shifter, style in ((demo, "--"), (graded_demo, "-")):
        profiles = [p for p in shifter.device.doping if p.region == name]
        acceptors, donors = acceptor_donor_concentrations(profiles, y[inside], 0.0)
        axes[0].semilogy(y[inside], np.maximum(donors, 1e14), "C0" + style)
        axes[0].semilogy(y[inside], np.maximum(acceptors, 1e14), "C3" + style)
axes[0].set_ylim(1e15, 1e18)
axes[0].set_xlabel("y (um)")
axes[0].set_ylabel("doping (cm$^{-3}$)")
axes[0].set_title("Donors (blue), acceptors (red)")

for label, s, style in (("abrupt", sweep, "o--"), ("graded", graded_sweep, "o-")):
    axes[1].plot(s.voltages, s.capacitance_f_per_m * 1e12, style, label=label)
axes[1].set_ylabel("C (pF/m)")
for label, o, v, style in (
    ("abrupt", optical, vpi_l, "o--"),
    ("graded", graded_optical, graded_vpi_l, "o-"),
):
    axes[2].plot(o.voltages, v, style, label=label)
axes[2].set_ylabel("$V_\\pi L$ (V cm)")
for ax in axes[1:]:
    ax.set_xlabel("Reverse bias (V)")
    ax.legend()
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

at = -2  # where V_pi L is quoted above
print(
    f"at {optical.voltages[at]:.1f} V:  "
    f"C {sweep.capacitance_f_per_m[at] * 1e12:.0f} -> "
    f"{graded_sweep.capacitance_f_per_m[at] * 1e12:.0f} pF/m,  "
    f"V_pi L {vpi_l[at]:.2f} -> {graded_vpi_l[at]:.2f} V cm,  "
    f"loss {optical.loss_db_cm[at]:.1f} -> {graded_optical.loss_db_cm[at]:.1f} dB/cm"
)

Dashed is the abrupt device, solid the graded one.

- The **capacitance is lower at every bias**, by 12 % at 0 V and 5 % at
  2 V (of a total that includes the oxide's fringing capacitance, which grading
  does not touch). With less net doping beside the junction, the same voltage depletes a
  wider strip, and a wider dielectric is a smaller capacitor. The gap is widest
  at low bias, where the whole depletion region sits inside the compensated
  zone; at higher bias its edges are out in silicon doped as drawn.
- **$V_\pi L$ is higher**, about 10 % at the quoted bias and 18 % at 0 V: the
  depletion edge now sweeps through silicon holding fewer carriers, so each
  volt removes fewer of them from the mode. This is the cost of grading. A lower
  capacitance loads the RF line less, which helps the bandwidth, and the loss
  drops a little, but the drive voltage is what a link budget notices first.
- The junction itself has moved. The two cores are not doped alike (p 5e17,
  n 3e17 cm⁻³), so the curves in the left-hand plot cross about 10 nm inside
  the n side: the heavier side's tail wins until there.

The **textbook check above applies to the abrupt device only**. The depletion
approximation assumes a step from one uniform doping to another, which is what
`PNJunctionConfig` describes; against the graded curve it would read well high
at low bias, and that would say nothing about the simulation. A graded junction
is compared with a measured $C(V)$, or with a process simulator's profile
brought in through `TableDoping`. The rest of the notebook continues with the
abrupt device.

### Continuous profile or staircase

femwell, the default optical solver, accepts a permittivity that varies from
element to element, so it uses the carrier map as is. Palace, the alternative
(`study.optical(route="palace", n_strips=...)`), accepts one constant material
per region only. For Palace the doped silicon is therefore cut into $N$ vertical
**strips**, each with the average properties of the carriers inside it: a
**staircase** approximation of the smooth profile.

The strip count matters more than one might expect. Measured on a simple
test device (`demo_phase_shifter`) against the continuous reference:

| strips | error on $\mathrm{Re}\,n_{eff}$ | error on $\Delta n_{eff}$ |
|---|---|---|
| 2 | 0.3 % | 59 % |
| 16 | 0.01 % | 2 % |

$n_{eff}$ depends on the whole waveguide and converges immediately;
$\Delta n_{eff}$, and therefore $V_\pi L$, depends on a sliver of silicon a few
tens of nanometres wide and needs many strips to resolve it. For a rib device
such as this one, prefer the continuous femwell route for the optical step: an
optical staircase draws every strip at the rib's full height, which thickens
the slab, and the Stage warns when asked to do that.

## Step 4: the RF transmission line

Now the electrical side. The two electrodes form a transmission line, and the
doped silicon between them loads it: the junction adds capacitance, and the
resistive silicon on either side of it adds loss. The RF Stage finds the
propagating mode of that loaded line and reads off its per-unit-length
behaviour.

### The staircase the RF Stage solves

Both RF solvers take one constant material per region, so here the staircase is
always used. The carrier map at one bias (the last of the sweep, 2 V, unless
you set `study.rf(bias_v=...)`) is reduced to strips across the whole doped
slab, each with the conductivity of its average carrier concentrations.

The strips follow the drawn device: none straddles two regions, and each has
its own region's height (220 nm in the rib, 90 nm in the slab). The rib, where
the conductivity changes quickly, gets $N$ strips (21 by default); each of the
other regions gets two.

The plot shows the result: highly conducting contact regions at both ends,
then the plus regions, the lightly doped core, and at the centre a few strips
with essentially zero conductivity. That insulating gap is the junction
capacitor; the conducting silicon on each side is its series resistance.

In [ ]:
staircase = study.rf.staircase()

edges = np.asarray(staircase.strips.edges_um, dtype=float)
sigma = np.asarray(staircase.strips.conductivity_s_per_m, dtype=float)
print("strip regions:  ", staircase.strip_names)
print("strip edges (um):", np.round(edges, 3))
print("strip sigma (S/m):", np.round(sigma, 1))
print("electrodes (um):", staircase.electrode_spans)
print("signal conductor:", study.rf.signal_electrode())

fig, ax = plt.subplots(figsize=(6, 3))
ax.stairs(sigma, edges, fill=True, alpha=0.6)
ax.set_xlabel("y (um)")
ax.set_ylabel("$\\sigma$ (S/m)")
ax.set_title(f"Staircase conductivity at V = {study.rf.bias_point().bias_v} V")
ax.grid(alpha=0.3)
plt.show()

### Line parameters versus frequency

The Stage meshes the staircase with its two electrodes once, then solves for
the line mode at each frequency. Three numbers come out per frequency:

- the **RF index** $n_{RF}$: how slowly the RF wave travels ($v = c/n_{RF}$);
- the **RF loss** $\alpha_{RF}$ in dB/cm;
- the **characteristic impedance** $Z_0 = 2P/|I|^2$, from the power carried by
  the mode and the current in the signal electrode.

These come directly from the field solution; no analytic transmission-line
formula is involved. The mode is tracked from one frequency to the next, so the
solver stays on the same mode across the sweep.

The same staircase can be solved with Palace (`study.rf(route="palace")`), which
reports the same quantities. One difference is worth knowing: by default the
femwell route meshes the electrodes as real metal with a finite conductivity,
whereas the Palace route treats them as perfect conductors. To compare the two
like for like, set `conductor_model=` to the same value on both.

In [ ]:
line_params = study.rf.run()

for freq, n_rf, alpha, z0 in zip(
    line_params.freq_hz,
    line_params.n_rf,
    line_params.alpha_rf_np_m,
    line_params.z0_ohm,
    strict=True,
):
    print(
        f"f = {freq / 1e9:5.0f} GHz:  n_RF = {n_rf:5.2f},  "
        f"alpha_RF = {alpha * 8.686 / 100:5.1f} dB/cm,  Z0 = {z0.real:5.1f} ohm"
    )
print(f"staircased at V = {line_params.bias_v} V")

Three things to read from this table:

- $n_{RF}$ is 3 to 4, against about 2 for the same electrodes without the
  junction. The junction capacitance **slows the wave down**, a *slow-wave*
  line, which is what brings the RF velocity close to the optical one
  ($n_g \approx 3.9$, computed in Step 5). The slowing fades at high frequency, where the junction
  capacitor no longer has time to charge through the resistive silicon.
- $Z_0$ stays between 26 and 32 Ω, for the same reason: extra capacitance per
  unit length lowers the impedance. Real designs space the electrodes further
  apart, or load the line only periodically, to get closer to 50 Ω.
- The RF loss grows quickly with frequency. The current that charges the
  junction capacitor flows through resistive silicon, and that charging current
  grows with frequency. This demonstration device is not optimised and is
  heavily loaded.

Below is the mesh the RF Stage solved: the strips coloured by conductivity,
between the two electrodes, in oxide. In the close-up the depleted strips at
the junction are the dark band at the centre.

In [ ]:
rf_mesh = study.stage_dir("rf") / "palace.msh"
strip_sigma = dict(
    zip(staircase.strip_names, staircase.strips.conductivity_s_per_m, strict=True)
)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, title, xlim, ylim in (
    (axes[0], "RF staircase: the whole window", None, None),
    (
        axes[1],
        "RF staircase: the strips",
        (edges[0] - 1.0, edges[-1] + 1.0),
        (-0.4, 1.2),
    ),
):
    plot_cross_section(
        rf_mesh,
        ax=ax,
        title=title,
        values=strip_sigma,
        label="$\\sigma$ (S/m)",
        xlim=xlim,
        ylim=ylim,
    )
plt.tight_layout()
plt.show()

## Step 5: the whole device

### Velocity mismatch

The light travels at the optical *group* velocity ($c/n_g$), the RF wave at
$c/n_{RF}$. If the two differ, the light gradually slips relative to the RF
wave and, over a long enough electrode, sees the drive voltage average out.
For a lossless, matched line this alone limits the bandwidth to

$$f_{3dB} = \frac{1.39\,c}{\pi\, L\, |n_{RF} - n_g|}$$

so a longer modulator (lower drive voltage) has a lower walk-off limit.

A single-wavelength mode solve gives the phase index, not the group index, so
the optical Stage solves the mode 10 nm either side of the wavelength, with
every material re-resolved there, and forms
$n_g = n_{eff} - \lambda\, dn_{eff}/d\lambda$ (`study.optical.group_index()`).
The line Stage uses it automatically; pass `study.line(n_group=...)` only to
substitute a measured value.

On a loaded line $n_{RF}$ falls with frequency, so the mismatch in the formula
is the one the line has *at the frequency in question*:
`walkoff_bandwidth_dispersive` solves for the frequency that satisfies the
condition with its own $n_{RF}(f)$. Beyond the last solved frequency it keeps
the last solved index, so values above the dotted line in the right-hand plot
are extrapolations.

On this device $n_{RF}$ starts above $n_g$ and ends below it. The walk-off
limit for the 3 mm electrode comes out well above the electro-optic bandwidth
found in the next section, so velocity mismatch is not what limits this
modulator.

In [ ]:
N_GROUP_OPT = study.optical.group_index()  # two more mode solves, kept with the sweep
group = study.optical.result.group_index
print(f"n_eff = {optical.n_eff[0].real:.4f}, n_g = {N_GROUP_OPT:.4f}")
print(
    f"  from n_eff = {group.n_eff[0]:.4f} / {group.n_eff[1]:.4f} at "
    f"{group.wavelengths_um[0]:g} / {group.wavelengths_um[1]:g} um; "
    f"material dispersion: {group.material_dispersion}"
)

In [ ]:
from gsim.modulator.twmzm import walkoff_bandwidth_dispersive

lengths_mm = np.linspace(1, 10, 40)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(line_params.freq_hz / 1e9, line_params.n_rf, "o-", label="$n_{RF}$ solved")
axes[0].axhline(N_GROUP_OPT, color="k", ls="--", label="$n_g$ optical")
axes[0].set_xlabel("frequency (GHz)")
axes[0].set_ylabel("index")
axes[0].set_title("Velocity matching vs frequency")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(
    lengths_mm,
    [
        (
            walkoff_bandwidth_dispersive(
                line_params.freq_hz,
                line_params.n_rf,
                length_m=length * 1e-3,
                n_opt=N_GROUP_OPT,
            )
            or np.nan
        )
        / 1e9
        for length in lengths_mm
    ],
)
axes[1].axhline(
    max(RF_FREQS_HZ) / 1e9, color="k", ls=":", label="last solved frequency"
)
axes[1].axvline(study.line.length_um * 1e-3, color="r", ls="--", label="this device")
axes[1].set_xlabel("electrode length (mm)")
axes[1].set_ylabel("walk-off $f_{3dB}$ (GHz)")
axes[1].set_title("Walk-off limit")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Electro-optic response and figures of merit

The line Stage adds what the cross-section cannot know: the electrode
**length**, the **source impedance** driving it and the **termination** at its
end. It then computes the standard small-signal traveling-wave response: the
RF voltage wave along the line, including attenuation and the reflections at
both ends, averaged over the transit of the light. Velocity mismatch, RF loss
and impedance mismatch are all included.

Here the line is driven from 50 Ω and terminated in its own characteristic
impedance, so nothing reflects from the far end. The response is evaluated on
a finer frequency grid than the RF Stage solved, by interpolation within the
solved range, so the 3 dB point can be read accurately. `study.report()` runs
any Stage that has not run yet; here they all have.

In [ ]:
study.line(
    z_load_ohm=float(np.mean(line_params.z0_ohm.real)),  # terminate in the line
    z_gen_ohm=50.0,
    # A denser grid than the RF stage solved, so the 3 dB point is
    # readable; kept inside the solved range, which is not extrapolated.
    response_frequencies_hz=list(np.linspace(min(RF_FREQS_HZ), max(RF_FREQS_HZ), 200)),
)
report = study.report()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(report.freq_hz / 1e9, 20 * np.log10(np.abs(report.response)))
ax.axhline(20 * np.log10(1 / np.sqrt(2)), color="k", ls=":", label="-3 dB (EO)")
if report.bandwidth_3db_hz:
    ax.axvline(report.bandwidth_3db_hz / 1e9, color="r", ls="--")
ax.set_xlabel("frequency (GHz)")
ax.set_ylabel("EO response (dB)")
ax.set_title(f"TW-MZM response, L = {report.length_m * 1e3:.0f} mm, matched load")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

print("=== TW-MZM figures of merit ===")
if report.bandwidth_3db_hz:
    print(f"EO 3 dB bandwidth:        {report.bandwidth_3db_hz / 1e9:.1f} GHz")
if report.walkoff_bandwidth_hz:
    print(f"walk-off limit:           {report.walkoff_bandwidth_hz / 1e9:.1f} GHz")
print(
    f"velocity mismatch:        n_RF - n_g = {report.velocity_mismatch[0]:+.2f} at "
    f"{report.freq_hz[0] / 1e9:.0f} GHz, {report.velocity_mismatch[-1]:+.2f} at "
    f"{report.freq_hz[-1] / 1e9:.0f} GHz"
)
print(f"characteristic impedance: {report.z0_ohm[0].real:.1f} ohm")
print(
    f"V_pi L at {report.voltages_v[-2]:.1f} V:        {report.vpi_l_vcm[-2]:.2f} V cm"
)
rlgc = report.rlgc
print(
    f"RLGC at 10 GHz: R = {np.interp(10e9, report.freq_hz, rlgc['R']):.0f} ohm/m, "
    f"L = {np.interp(10e9, report.freq_hz, rlgc['L']) * 1e9:.0f} nH/m, "
    f"C = {np.interp(10e9, report.freq_hz, rlgc['C']) * 1e12:.0f} pF/m"
)

The electro-optic bandwidth is a few tens of GHz, less than half the walk-off
limit: this device is limited by **RF loss**. The way to improve it is to reduce the
series resistance of the silicon or the loading of the line, not to tune the
velocity match.

The report also gives the line's **RLGC** parameters (series resistance and
inductance, shunt conductance and capacitance per unit length), derived from
$\gamma$ and $Z_0$. They are what a circuit simulator needs. To take the
results elsewhere, `study.line.export_touchstone(...)` writes the electrode as
a two-port S-parameter file and `study.charge.export_junction_model()` writes
the junction's $C_j(V)$ and $R_s(V)$.

### From phase shifter to modulator: transfer, $V_\pi$, extinction and chirp

So far the report describes the phase shifter. A modulator puts one in each arm
of a Mach-Zehnder interferometer, and what you measure on the bench is the
optical power out against the drive voltage. The line Stage computes that
static transfer $T(v)$ from the optical bias sweep (index shift and loss versus
bias) and the electrode length, and reads the datasheet numbers off it:

- **$V_\pi$**: the drive swing from the peak of the transfer to the adjacent null,
  at this length (not the length-independent $V_\pi L$ above);
- **insertion loss**: the transmission at the peak;
- **extinction ratio**: peak over null. It is finite because the two arms sit at
  different voltages at the null, so their free-carrier losses differ and the
  fields do not cancel, and because a real splitter is never exactly 50:50
  (`arm_imbalance_db`);
- **chirp**: the small-signal chirp parameter at each bias point, from the index
  and absorption slopes. Positive means the optical frequency rises while the
  intensity rises.

The drive voltage $v$ is the voltage between the two arms. `"single-drive"` puts
all of it on one arm and leaves the other at the arm bias; `"push-pull"` moves
the arms by $\pm v/2$ about it. Both arms rest in the middle of the bias sweep
unless you set `arm_bias_v`, and the interferometer is parked at quadrature
(`phase_offset_rad`), where it rests at half its peak transmission. Nothing is
extrapolated: the transfer only covers the drive the bias sweep can answer for,
which is twice as wide for push-pull. If the sweep is too short to go from a
peak to a null, $V_\pi$, insertion loss and extinction ratio are left out and
`transfer_message` says why; sweep more bias or lengthen the electrode.
Re-configuring the line Stage does not re-run any solver.

That last point bites here: with $V_\pi L$ near 2.1 V cm, the 3 mm electrode
has a $V_\pi$ close to 7 V, and the 0 to 2 V bias sweep cannot swing either
configuration from a peak to a null. So this section looks at a **6 mm**
electrode, the length at which a push-pull drive fits inside the sweep; the
rest of the notebook stays at 3 mm.

In [ ]:
MZ_LENGTH_UM = 6000.0  # long enough for a push-pull V_pi inside the 0-2 V sweep

mz_reports = {}
for drive in ("single-drive", "push-pull"):
    # a 0.2 dB splitter imbalance
    study.line(length_um=MZ_LENGTH_UM, drive=drive, arm_imbalance_db=0.2)
    mz_reports[drive] = study.report()
# back to the 3 mm device and the defaults
study.line(length_um=LENGTH_UM, drive="push-pull", arm_imbalance_db=0.0)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for drive, rep in mz_reports.items():
    axes[0].plot(rep.drive_v, 10 * np.log10(rep.transfer), label=drive)
    axes[1].plot(rep.voltages_v, rep.chirp, "o-", label=drive)
axes[0].set_xlabel("drive voltage between the arms (V)")
axes[0].set_ylabel("transmission (dB)")
axes[0].set_ylim(-40, 0)
axes[0].set_title(f"Static transfer, L = {MZ_LENGTH_UM * 1e-3:.0f} mm, at quadrature")
axes[1].set_xlabel("Reverse bias (V)")
axes[1].set_ylabel("chirp parameter")
axes[1].set_title("Small-signal chirp")
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

for drive, rep in mz_reports.items():
    print(f"=== {drive} (arms resting at {rep.arm_bias_v:.1f} V) ===")
    if rep.v_pi_v is None:
        print(rep.transfer_message)
    else:
        print(f"V_pi:              {rep.v_pi_v:.2f} V")
        print(f"insertion loss:    {rep.insertion_loss_db:.1f} dB")
        print(f"extinction ratio:  {rep.extinction_ratio_db:.1f} dB")
    print(
        f"chirp:             {rep.chirp[0]:+.2f} to {rep.chirp[-1]:+.2f} along the sweep"
    )

Single-drive chirp sits near $+1$ at this quadrature point
(`phase_offset_rad=-np.pi/2` gives $-1$); push-pull cancels it down to the
small residue left by the two arms' losses moving in opposite directions and by
the splitter imbalance. Push-pull also reaches $V_\pi$ from a bias sweep half as
wide, which is why it reports its figures at 6 mm while the single drive still
prints the "too short" message: its sweep does not cover a full peak-to-null
swing at this length.

### Reaching toward 50 Ω: a segmented electrode

Loaded all the way by the junction, this line sits near 26 Ω at 20 GHz with an
RF index of 3.7. Real modulators close the gap to 50 Ω by loading the electrode
only part of the way: short **loaded** sections alternate with **unloaded**
ones along the line, with a period far below the RF wavelength. The RF wave
then sees a periodic line whose impedance and index lie between the two — its
*Bloch* impedance and index — while the light is modulated along the loaded
fraction only, so $V_\pi L$ of the device grows as one over the fill factor.

`study.line(fill_factor=..., period_um=...)` reports that device. Its second
ingredient is the bare electrode, which the RF Stage solves with the carriers
switched off (`study.rf.run_unloaded()`, run automatically the first time it
is needed). Each period is cascaded as two-ports from the two cross-section
solves. This is an approximation of a 3D structure: **both sections share one
electrode cross-section, with no loading fins drawn**, and the fields at the
section boundaries are not solved. The Stage warns if the period approaches
the Bragg condition inside the band. The bare line's two slowest modes sit
close together, so its solve may warn that the mode selection is degenerate
at the highest frequencies; the selected index stays smooth across the band.

In [ ]:
FILL_FACTORS = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3]
matched_load = study.line.z_load_ohm
segmented_reports = {}
for fill in FILL_FACTORS:
    # 50 ohm at both ends: the impedance a segmented electrode is built to reach.
    study.line(fill_factor=fill, period_um=50.0, z_load_ohm=50.0, z_gen_ohm=50.0)
    segmented_reports[fill] = study.report()
study.line(
    fill_factor=1.0, z_load_ohm=matched_load
)  # back to the electrode loaded all the way

grid = segmented_reports[1.0].freq_hz
i20 = int(np.argmin(np.abs(grid - 20e9)))
z_bloch = [abs(segmented_reports[f].z0_ohm[i20]) for f in FILL_FACTORS]
n_rf = [segmented_reports[f].n_rf[i20] for f in FILL_FACTORS]
bandwidth = [
    (segmented_reports[f].bandwidth_3db_hz or np.nan) / 1e9 for f in FILL_FACTORS
]
vpi_l_device = [segmented_reports[f].vpi_l_vcm[-2] for f in FILL_FACTORS]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].plot(FILL_FACTORS, z_bloch, "o-")
axes[0].axhline(50.0, color="k", ls=":", label="50 Ω")
axes[0].set_ylabel("$|Z_{Bloch}|$ at 20 GHz (Ω)")
axes[0].legend()
axes[1].plot(FILL_FACTORS, bandwidth, "o-")
axes[1].set_ylabel("EO 3 dB bandwidth (GHz)")
axes[2].plot(FILL_FACTORS, vpi_l_device, "o-")
axes[2].set_ylabel(f"device $V_\\pi L$ at {report.voltages_v[-2]:.1f} V (V cm)")
for ax in axes:
    ax.set_xlabel("fill factor")
    ax.grid(alpha=0.3)
fig.suptitle("Segmented electrode, 50 µm period, 50 Ω source and load")
plt.tight_layout()
plt.show()

for f, z, n, bw, v in zip(
    FILL_FACTORS, z_bloch, n_rf, bandwidth, vpi_l_device, strict=True
):
    print(
        f"fill {f:.1f}: |Z| = {z:5.1f} ohm, n_RF = {n:.2f}, "
        f"EO 3 dB = {bw:5.1f} GHz, V_pi L = {v:.2f} V cm"
    )

Lowering the fill factor from 1 to 0.3 lifts the impedance from about 26 to
36 Ω and the EO bandwidth from 18.5 to 23 GHz, and costs a factor 3.3 in
$V_\pi L$. It reaches *toward* the bare electrode, never past it, and this bare
electrode is itself only 45 to 47 Ω across the band: no fill factor lands on
50 Ω. To get there the unloaded electrode must be designed well above 50 Ω
(narrower signal, wider gap). The bandwidth gain is modest because this device
is limited by RF loss, and the walk-off limit actually falls as $n_{RF}$ drops
below the optical group index.

## Cross-check: the loaded line, computed two ways

Commercial workflows usually do not solve the loaded cross-section directly as
was done above. They proceed in two steps:

1. solve the **bare electrode**, with no carriers in the silicon, and extract
   its RLGC parameters;
2. add the junction afterwards as a lumped **series RC branch** per unit
   length, the junction capacitance $C_j$ in series with the silicon's
   resistance $R_s$, both taken from the charge simulation:
   $Y' = G' + i\omega C' + \dfrac{i\omega C_j}{1 + i\omega R_s C_j}$.

The branch is distributed along the line, not attached at one port: the
junction loads the electrode along its whole length, and that continuous
loading is what produces the slow wave.

`study.rf.crosscheck()` runs both methods at the same bias and compares them.
For a fair comparison the direct solve needs a finer staircase (61 strips
across the rib instead of 21) and a mode search aimed at the slow wave. The
comparison is made at 20 and 30 GHz.

In [ ]:
study.rf(
    frequencies_hz=[20e9, 30e9],
    n_strips=61,
    num_modes=8,
    n_guess=4.5,
    # The bare line's slowest two modes sit 1.5 % apart; the Stage's
    # wall-mode check, not the index, is what tells the line mode apart.
    degeneracy_rtol=0.01,
)
comparison = study.rf.crosscheck()

for i, freq in enumerate(comparison.freq_hz):
    print(
        f"f = {freq / 1e9:4.0f} GHz:  "
        f"n_RF {comparison.direct.n_rf[i]:5.2f} vs "
        f"{comparison.assembled.n_rf[i]:5.2f} ({comparison.delta_n_rf[i]:+.1%}),  "
        f"loss {comparison.delta_alpha[i]:+.1%},  |Z0| {comparison.delta_z0[i]:+.1%}"
    )
comparison.check()
print("cross-check passed: the two methods agree within the expected gap")

The two cross-sections behind the comparison: the loaded staircase, now with 61
strips across the rib so that several fall inside the depletion region, and the
same geometry with all carriers removed, which is the bare electrode of step 1.

In [ ]:
fine = study.rf.staircase()
fine_sigma = dict(zip(fine.strip_names, fine.strips.conductivity_s_per_m, strict=True))
fine_limits = (1e-2, max(fine_sigma.values()))  # one scale for both
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
for ax, subdir, title, sigma in (
    (axes[0], "", "Loaded: 61 strips", fine_sigma),
    (axes[1], "unloaded", "Unloaded: carriers off", dict.fromkeys(fine_sigma, 0.0)),
):
    plot_cross_section(
        study.stage_dir("rf") / subdir / "palace.msh",
        ax=ax,
        title=title,
        values=sigma,
        label="$\\sigma$ (S/m)",
        limits=fine_limits,
        xlim=(fine.strips.edges_um[0] - 1.0, fine.strips.edges_um[-1] + 1.0),
        ylim=(-0.4, 1.2),
    )
plt.tight_layout()
plt.show()

In [ ]:
unloaded = study.rf.run_unloaded()
direct, assembled = comparison.direct, comparison.assembled
freqs_ghz = comparison.freq_hz / 1e9

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, direct_y, assembled_y, unloaded_y, label in (
    (axes[0], direct.n_rf, assembled.n_rf, unloaded.n_rf, "$n_{RF}$"),
    (
        axes[1],
        direct.alpha_rf_np_m * 8.686 / 100,
        assembled.alpha_rf_np_m * 8.686 / 100,
        unloaded.alpha_rf_np_m * 8.686 / 100,
        "RF loss (dB/cm)",
    ),
    (
        axes[2],
        np.abs(direct.z0_ohm),
        np.abs(assembled.z0_ohm),
        np.abs(unloaded.z0_ohm),
        "$|Z_0|$ (ohm)",
    ),
):
    ax.plot(freqs_ghz, direct_y, "o-", label="direct loaded solve")
    ax.plot(freqs_ghz, assembled_y, "s--", label="unloaded + junction branch")
    ax.plot(freqs_ghz, unloaded_y, "^:", color="gray", label="unloaded electrode")
    ax.set_xlabel("frequency (GHz)")
    ax.set_ylabel(label)
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
fig.suptitle(f"The loaded line assembled two ways, at V = {comparison.direct.bias_v} V")
plt.tight_layout()
plt.show()

Both methods tell the same story: compared with the bare electrode (grey), the
loaded line is about twice as slow, several times more lossy, and has about
half the impedance.

They agree on the wave and not on its loss. The two-step method reads 2 to 5 %
low on $n_{RF}$ and 10 % high on $|Z_0|$, while the series resistance and
inductance agree to 1 %. That agreement rests on the oxide in the charge solve:
the field that fringes through the oxide above and below the junction, between
the two conducting halves of the silicon, is part of the capacitance the line
sees. A charge solve of the silicon alone (`study.charge(oxide=False)`) gives
160 pF/m for $C_j$ at 2 V instead of 270, and the two methods are then 16 to
19 % apart on $n_{RF}$ and 25 to 28 % on $|Z_0|$.

The loss is another matter: the two-step method reads about 38 % low, with the
capacitance matched, and the direct solve finds two to three times its shunt
conductance. A single series RC branch, fitted at 1 GHz, standing in for a
distributed slab at 20 to 30 GHz is one candidate; it has not been tested. The
loss gap has no confirmed explanation, and the direct solve is the reference.

`comparison.check()` raises an error if the two methods ever disagree by more
than this known gap, naming the quantity and the frequency. If you rely on
lumped junction models in a circuit simulator, this figure is the size of the
error to expect, and the direct solve is the reference.

## Summary

Starting from one gdsfactory component, one layer stack and a description
saying which regions are p and which are n:

- **DEVSIM** solved the carrier distribution at each bias and the junction
  capacitance: the textbook formula for the depleted silicon, plus the 100 to
  120 pF/m that fringes through the oxide around it;
- the **carriers** Stage converted those maps into an optical index change,
  an optical absorption and an RF conductivity;
- **femwell** solved the optical mode on the continuous carrier profile, giving
  $\Delta n_{eff}(V)$, the optical loss and $V_\pi L$, and the RF mode of the
  loaded electrodes, giving $n_{RF}$, the RF loss and $Z_0$;
- the **line** Stage combined them into the electro-optic response, its 3 dB
  bandwidth, and the RLGC parameters.

Things to try from here:

- your own device: pass your `component`, `stack` and `device` to
  `pn_phase_shifter`;
- another wavelength: `wavelength_um=1.31` selects the matching
  plasma-dispersion coefficients;
- a different junction: change the doping in the device description and watch
  $V_\pi L$ trade against loss. Only the Stages that depend on the change run
  again;
- a different electrode: change its geometry or `length_um`, and see $Z_0$ and
  the bandwidth move;
- another solver: `study.rf(route="palace")`.